#### Setup: catalog, schemas, volume, and tables
Creates the Unity Catalog storage used by the whole pipeline. Run once before any other notebook.

**Creates**
- Catalog: `brp`
- Schemas (medallion layers):
  - `brp.bronze`: raw data exactly as received
  - `brp.silver`: cleaned, structured closure records
  - `brp.gold`: map-ready outputs and daily summaries
- Volume: `brp.bronze.raw`, raw downloaded files (road status HTML, milepost GeoJSON)
- Tables:
  - `brp.bronze.road_status_raw`: one row per saved download of the road status page (several per day possible)
  - `brp.silver.road_sections_status`: the page parsed into road sections, one row per section per download
  - `brp.silver.mileposts`: NPS Blue Ridge Parkway milepost points (one per whole milepost)'
**Steps**
1. Create the catalog
2. Create the three medallion schemas
3. Create the raw-file volume
4. Create the bronze road status table
5. Create the silver road sections table
6. Check: list schemas, tables and volumes

**Safe to re-run:** every statement uses `IF NOT EXISTS`, so running it again never changes or deletes existing data. This also means changes to an existing table's columns or comments aren't applied on a rerun; use `ALTER TABLE` / `COMMENT ON`, or drop and recreate the table.

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: CATALOG
-- Creates `brp`, the top-level container for everything in the Blue Ridge
-- Parkway closures project.
--
-- Unity Catalog organizes data in three levels:
--   catalog  ->  schema  ->  table / volume
--   brp      ->  bronze  ->  road_status_raw
-- Written as brp.bronze.road_status_raw throughout the project notebooks.
--
-- The bronze, silver and gold schemas (one per medallion layer) are created
-- inside this catalog in the next cells.
--
-- IF NOT EXISTS makes this safe to rerun: if the catalog is already there,
-- nothing happens, and nothing inside it is changed or deleted.
--
-- Note: creating a catalog needs the CREATE CATALOG permission on the
-- metastore. If this fails with a permissions error, a workspace admin
-- needs to grant it.
-- ---------------------------------------------------------------------------
CREATE CATALOG IF NOT EXISTS brp
COMMENT 'Blue Ridge Parkway closures project: bronze, silver and gold layers';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SCHEMAS (one per medallion layer)
-- Creates the three schemas inside the brp catalog. Each schema holds one
-- layer of the medallion architecture, and data flows bronze -> silver -> gold:
--
--   bronze  Raw data, exactly as downloaded. Never edited, so later layers
--           can always be rebuilt from it.
--             e.g. brp.bronze.road_status_raw  (one row per page download)
--                  brp.bronze.raw              (volume holding the HTML files)
--
--   silver  Cleaned and structured: parsed into proper columns and types,
--           with status values standardized.
--             e.g. brp.silver.road_sections_status  (one row per road section)
--
--   gold    Summaries built from silver, ready for reports and dashboards
--           (e.g. miles closed per day, closure history by section).
--
-- IF NOT EXISTS makes this safe to rerun: existing schemas, and the tables
-- inside them, are left untouched.
-- ---------------------------------------------------------------------------

CREATE SCHEMA IF NOT EXISTS brp.bronze
COMMENT 'Raw data, exactly as received from the source';

CREATE SCHEMA IF NOT EXISTS brp.silver
COMMENT 'Cleaned, standardized data parsed from bronze';

CREATE SCHEMA IF NOT EXISTS brp.gold
COMMENT 'Aggregated datasets for reporting and dashboards';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: BRONZE VOLUME
-- File storage for raw downloads (road status HTML, milepost GeoJSON),
-- saved exactly as received before silver parses them.
-- In Python, it's the folder: /Volumes/brp/bronze/raw
-- IF NOT EXISTS: safe to rerun; existing files are left untouched.
-- ---------------------------------------------------------------------------
CREATE VOLUME IF NOT EXISTS brp.bronze.raw
COMMENT 'Raw source files exactly as downloaded: road status HTML and milepost GeoJSON';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- BRONZE TABLE: brp.bronze.road_status_raw
-- One row per download of the NPS road status page. The job can run several
-- times a day; unchanged pages are skipped, so each row is a new version of
-- the page. Keeps the raw HTML plus download details, so silver can always be
-- rebuilt from here.
--
-- CREATE TABLE IF NOT EXISTS only creates the table the first time. If it
-- already exists, this statement does nothing (it won't change or empty it).
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.bronze.road_status_raw (
  source      STRING    COMMENT 'Which source this came from, e.g. road_status_page',
  url         STRING    COMMENT 'Address that was downloaded',
  run_date    DATE      COMMENT 'Pipeline day in America/New_York; several rows can share a day',
  fetched_at  TIMESTAMP COMMENT 'Exact download time (UTC); unique per download',
  http_status INT       COMMENT 'Server reply code, 200 = OK',
  sha256      STRING    COMMENT 'Fingerprint of the raw bytes; used to skip unchanged pages',
  file_path   STRING    COMMENT 'Where the raw file is saved in the volume',
  raw_html    STRING    COMMENT 'Full page HTML as received'
)
COMMENT 'Bronze: one raw snapshot of the road status page per download (several per day possible)';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SILVER TABLE
-- The road status page parsed into clean columns: one row per parkway
-- section per saved download (several downloads per day are possible).
-- Built by the silver notebook from brp.bronze.road_status_raw.
-- Columns must match SILVER_SCHEMA in the silver notebook.
-- IF NOT EXISTS: safe to rerun; an existing table and its data are untouched.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.silver.road_sections_status (
  -- When the data was captured
  run_date      DATE      COMMENT 'Pipeline day (America/New_York)',
  fetched_at    TIMESTAMP COMMENT 'Exact download time (UTC); links to the bronze row',
  nps_as_of     STRING    COMMENT 'NPS "Road status as of" text from the page',

  -- Where the section is
  state         STRING    COMMENT 'VA or NC',
  section_order INT       COMMENT 'Row order on the page (mileposts increase north to south)',
  mp_start      DOUBLE    COMMENT 'Lower milepost of the section',
  mp_end        DOUBLE    COMMENT 'Higher milepost of the section',
  length_mi     DOUBLE    COMMENT 'Section length in miles (mp_end - mp_start)',
  crossroads    STRING    COMMENT 'Roads or landmarks at each end of the section',

  -- What the status is
  status_raw    STRING    COMMENT 'Status exactly as written by NPS',
  notes         STRING    COMMENT 'Notes / maintenance information column',
  status        STRING    COMMENT 'Classified: open, reduced, closed, unknown',
  gated         BOOLEAN   COMMENT 'False when NPS lists the section as Ungated'
)
COMMENT 'Silver: Blue Ridge Parkway road status by section, one row per section per download';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- PROJECT SETUP: SILVER MILEPOSTS (reference table)
-- One row per whole milepost along the parkway (0 to 469), each with its
-- map coordinates. Loaded from the NPS ArcGIS milepost data.
--
-- Used to place road sections on a map: a section like 291.8 - 305.1 is
-- drawn by looking up the points for its start and end mileposts, and
-- estimating positions between whole mileposts where needed.
--
-- Reference data: it rarely changes, unlike the road status table.
-- IF NOT EXISTS: safe to rerun; an existing table and its data are untouched.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.silver.mileposts (
  milepost    DOUBLE    COMMENT 'Parkway milepost (whole miles, 0 to 469); DOUBLE to match mp_start / mp_end in road_sections_status',
  longitude   DOUBLE    COMMENT 'Longitude (east-west position), WGS84 / EPSG:4326, the standard GPS coordinate system',
  latitude    DOUBLE    COMMENT 'Latitude (north-south position), WGS84 / EPSG:4326',
  fetched_at  TIMESTAMP COMMENT 'When the points were downloaded (UTC)'
)
COMMENT 'Silver reference: NPS Blue Ridge Parkway milepost points (one per whole milepost)';

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- GOLD: SECTION STATUS (map-ready)
-- Every road section from silver, plus its line shape so it can be drawn
-- on a map and colored by status.
-- One row per section per saved download (several downloads per day possible).
-- Built from: brp.silver.road_sections_status + brp.silver.mileposts
-- IF NOT EXISTS: safe to rerun; an existing table and its data are untouched.
-- ---------------------------------------------------------------------------
CREATE TABLE IF NOT EXISTS brp.gold.section_status (
  -- When the data was captured
  run_date          DATE      COMMENT 'Pipeline day (America/New_York)',
  fetched_at        TIMESTAMP COMMENT 'Exact download time (UTC); links to the silver and bronze rows',
  nps_as_of         STRING    COMMENT 'When NPS last updated the page ("Road status as of" text)',

  -- Where the section is
  state             STRING    COMMENT 'VA or NC',
  section_order     INT       COMMENT 'Row order on the page (mileposts increase north to south)',
  mp_start          DOUBLE    COMMENT 'Lower milepost of the section',
  mp_end            DOUBLE    COMMENT 'Higher milepost of the section',
  length_mi         DOUBLE    COMMENT 'Section length in miles',
  crossroads        STRING    COMMENT 'Roads or landmarks at each end of the section',

  -- What the status is
  status            STRING    COMMENT 'Our category: open, reduced, closed, unknown',
  status_raw        STRING    COMMENT 'Status exactly as written by NPS',
  notes             STRING    COMMENT 'NPS notes / maintenance information',
  gated             BOOLEAN   COMMENT 'False when NPS lists the section as Ungated',

  -- Map shape
  point_count       INT       COMMENT 'How many points make up the line (very low = suspect shape)',
  geometry_geojson  STRING    COMMENT 'The section''s line as GeoJSON text: [longitude, latitude] points in GPS coordinates (WGS84 / EPSG:4326)'
)
COMMENT 'Gold: map-ready road sections with status and line shape, one row per section per download';

In [0]:
# ---------------------------------------------------------------------------
# DAILY SUMMARY: MILES BY STATUS FOR THE WHOLE PARKWAY AND EACH STATE
# Builds 3 rows (ALL, VA, NC) from the latest download in gold.section_status,
# then saves them as TODAY's summary, replacing any earlier summary for the
# same day. So the table always keeps one set of rows per day, from that
# day's last download (one point per day on trend charts).
#
# GROUPING SETS computes two groupings in one query:
#   (run_date, fetched_at)        -> one row for the whole parkway (state is NULL, so scope = 'ALL')
#   (run_date, fetched_at, state) -> one row per state (scope = 'VA' / 'NC')
# ---------------------------------------------------------------------------

summary_df = spark.sql(f"""
    SELECT
      run_date,
      fetched_at,
      MAX(nps_as_of) AS nps_as_of,   -- same text on every row of one download; MAX just picks it

      -- GROUPING(state) = 1 on the whole-parkway row (state was not grouped), else 0
      CASE WHEN GROUPING(state) = 1 THEN 'ALL' ELSE state END AS scope,

      -- COUNT returns a BIGINT; cast to match the table's INT column
      CAST(COUNT(*) AS INT) AS sections,

      -- Miles by status: add up length_mi, counting only sections with that status
      ROUND(SUM(length_mi), 1)                                                AS miles_total,
      ROUND(SUM(CASE WHEN status = 'open'    THEN length_mi ELSE 0 END), 1)   AS miles_open,
      ROUND(SUM(CASE WHEN status = 'reduced' THEN length_mi ELSE 0 END), 1)   AS miles_reduced,
      ROUND(SUM(CASE WHEN status = 'closed'  THEN length_mi ELSE 0 END), 1)   AS miles_closed,
      ROUND(SUM(CASE WHEN status = 'unknown' THEN length_mi ELSE 0 END), 1)   AS miles_unknown,

      -- Percentages (0-100) of total miles
      ROUND(100 * SUM(CASE WHEN status = 'open' THEN length_mi ELSE 0 END)
                / SUM(length_mi), 1)                                          AS pct_open,
      ROUND(100 * SUM(CASE WHEN status IN ('open', 'reduced') THEN length_mi ELSE 0 END)
                / SUM(length_mi), 1)                                          AS pct_passable   -- drivable miles

    FROM {GOLD_SECTIONS}
    WHERE fetched_at = TIMESTAMP '{fetched_at}'     -- ONLY the latest download, never a whole day
    GROUP BY GROUPING SETS (
      (run_date, fetched_at),                       -- whole parkway
      (run_date, fetched_at, state)                 -- each state
    )
""")

# Save as today's summary: overwrite, but ONLY this run_date's rows.
# A later download the same day replaces these 3 rows; other days are untouched.
(summary_df.write
    .mode("overwrite")
    .option("replaceWhere", f"run_date = '{run_date}'")
    .saveAsTable(GOLD_SUMMARY))

display(summary_df)

In [0]:
%sql
-- ---------------------------------------------------------------------------
-- SETUP CHECK: WHAT'S IN THE brp CATALOG
-- Lists every schema and the tables inside it, to confirm the medallion
-- setup (bronze, silver, gold) was created correctly.
--
-- Expected: bronze, silver and gold all appear. A schema with no tables yet
-- (e.g. gold early on) still shows, with NULL in the table columns.
-- ---------------------------------------------------------------------------
SELECT
    schemata.catalog_name AS table_catalog,   -- always brp here
    schemata.schema_name  AS table_schema,    -- bronze, silver, gold
    tables.table_name,                        -- a table in that schema (NULL if it has none)
    tables.table_type                         -- MANAGED (Databricks stores the data), EXTERNAL, VIEW...

-- information_schema is a built-in set of read-only views describing what's
-- in the catalog: `schemata` lists schemas, `tables` lists tables.
FROM brp.information_schema.schemata

-- LEFT JOIN keeps every schema even if it has no tables.
-- (A regular JOIN would hide empty schemas, like a new gold layer.)
LEFT JOIN brp.information_schema.tables
    ON  schemata.catalog_name = tables.table_catalog
    AND schemata.schema_name  = tables.table_schema

-- Hide the two schemas Databricks creates automatically in every catalog:
--   information_schema = the built-in metadata views used above
--   default            = an empty starter schema this project doesn't use
WHERE schemata.schema_name NOT IN ('information_schema', 'default')

-- Group the list by schema, then alphabetically by table.
ORDER BY schemata.schema_name, tables.table_name